# Covariance-repair sensitivity

Eigenvalue clipping can restore positive-semidefinite structure after an inconsistent correlation shock. It cannot recover the unknown population covariance. This notebook measures how the numerical projection changes a constrained allocation.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd

from portfolio_optimizer import (
    StaticOptimizer,
    create_style_constraints,
    nearest_psd,
    portfolio_performance,
)

## Construct a valid baseline and an invalid stressed matrix

In [ ]:
assets = ["Equity A", "Equity B", "Property", "Bonds", "Cash"]
growth_assets = assets[:3]
expected_returns = np.array([0.10, 0.08, 0.09, 0.04, 0.02])
volatilities = np.array([0.15, 0.12, 0.18, 0.05, 0.01])
correlation = np.array([
    [1.00, 0.75, 0.60, 0.10, 0.01],
    [0.75, 1.00, 0.50, 0.15, 0.02],
    [0.60, 0.50, 1.00, 0.20, 0.01],
    [0.10, 0.15, 0.20, 1.00, 0.05],
    [0.01, 0.02, 0.01, 0.05, 1.00],
])
scale = np.diag(volatilities)
covariance_baseline = scale @ correlation @ scale

correlation_stressed = correlation.copy()
correlation_stressed[0, 2] = correlation_stressed[2, 0] = -0.90
covariance_stressed = scale @ correlation_stressed @ scale
covariance_projected = nearest_psd(covariance_stressed, eigenvalue_floor=1e-12)

diagnostics = pd.DataFrame({
    "Minimum eigenvalue": [
        np.linalg.eigvalsh(covariance_baseline).min(),
        np.linalg.eigvalsh(covariance_stressed).min(),
        np.linalg.eigvalsh(covariance_projected).min(),
    ],
}, index=["Baseline", "Stressed input", "PSD projection"])
diagnostics["Frobenius distance from stressed input"] = [
    np.linalg.norm(covariance_baseline - covariance_stressed, ord="fro"),
    0.0,
    np.linalg.norm(covariance_projected - covariance_stressed, ord="fro"),
]
display(diagnostics)

## Compare constrained target-return allocations

Both optimisations use valid PSD matrices. Differences therefore measure sensitivity to the repair, not evidence that the projected matrix is a better risk estimate.

In [ ]:
bounds = tuple((0.0, 0.40) for _ in assets)
constraints = create_style_constraints(assets, growth_assets, lower=0.65, upper=0.75)
target_return = 0.0582
risk_free_rate = 0.02

optimizers = {
    "Baseline covariance": StaticOptimizer(expected_returns, covariance_baseline),
    "PSD-projected covariance": StaticOptimizer(expected_returns, covariance_projected),
}
weights = {}
rows = []
for name, optimizer in optimizers.items():
    result = optimizer.target_return(
        target_return,
        bounds=bounds,
        constraints=constraints,
    )
    weights[name] = result.x
    expected_return, volatility = portfolio_performance(
        result.x,
        optimizer.expected_returns,
        optimizer.covariance,
    )
    rows.append({
        "Covariance input": name,
        "Return": expected_return,
        "Volatility": volatility,
        "Sharpe": (expected_return - risk_free_rate) / volatility,
        "Max violation": result.max_constraint_violation,
    })

performance = pd.DataFrame(rows).set_index("Covariance input")
formatted = performance.copy()
formatted["Return"] = formatted["Return"].map(lambda value: f"{value:.2%}")
formatted["Volatility"] = formatted["Volatility"].map(lambda value: f"{value:.2%}")
formatted["Sharpe"] = formatted["Sharpe"].map(lambda value: f"{value:.4f}")
formatted["Max violation"] = formatted["Max violation"].map(lambda value: f"{value:.2e}")
display(formatted)

In [ ]:
weights_table = pd.DataFrame(weights, index=assets)
axis = weights_table.plot(kind="bar", figsize=(10, 5), width=0.75)
axis.set_title("Allocation sensitivity to covariance projection")
axis.set_ylabel("Portfolio weight")
axis.yaxis.set_major_formatter(PercentFormatter(1.0))
axis.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()